#Initial inspection

In [2]:
import sys
print(sys.executable)

/Users/jonsellers/Documents/transit_project_2026/.venv/bin/python


In [3]:
import boto3
import pandas as pd
from io import BytesIO

s3 = boto3.client("s3")

BUCKET_NAME = "jolese-transit-ml-portfolio-367995857052-us-east-1-an"
INTEGRATED_KEY = "integrated/monthly_base/run_date=2026-05-08/integrated_monthly_base.csv"

obj = s3.get_object(Bucket=BUCKET_NAME, Key=INTEGRATED_KEY)
df_integrated = pd.read_csv(BytesIO(obj["Body"].read()))

date_cols = [
    "date",
    "period_end",
    "transit_available_at",
    "gas_available_at",
    "inflation_available_at",
    "data_available_at",
]
for col in date_cols:
    if col in df_integrated.columns:
        df_integrated[col] = pd.to_datetime(df_integrated[col], errors="coerce")

print(df_integrated.shape)
print(df_integrated.head())
print(df_integrated.tail())
print(df_integrated.isna().sum())

/Users/jonsellers/Documents/transit_project_2026/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


(291, 21)
        date period_end transit_available_at      transit_series_id  \
0 2002-01-01 2002-01-31           2002-02-07  king_county_mb_do_bus   
1 2002-02-01 2002-02-28           2002-03-07  king_county_mb_do_bus   
2 2002-03-01 2002-03-31           2002-04-07  king_county_mb_do_bus   
3 2002-04-01 2002-04-30           2002-05-07  king_county_mb_do_bus   
4 2002-05-01 2002-05-31           2002-06-07  king_county_mb_do_bus   

                       transit_source_name        upt        vrm       vrh  \
0  ntd_complete_monthly_ridership_workbook  6045861.0  2878549.0  216788.0   
1  ntd_complete_monthly_ridership_workbook  5406135.0  2558164.0  192360.0   
2  ntd_complete_monthly_ridership_workbook  5999230.0  2840121.0  213519.0   
3  ntd_complete_monthly_ridership_workbook  6058398.0  2855047.0  213860.0   
4  ntd_complete_monthly_ridership_workbook  6134503.0  2899399.0  217418.0   

    voms gas_available_at  ... gas_source_name seattle_gas_price_avg  \
0  931.0              

In [4]:
print("Date range:", df_integrated["date"].min(), "to", df_integrated["date"].max())
print("Duplicate dates:", df_integrated["date"].duplicated().sum())

print("\nColumns:")
print(df_integrated.columns.tolist())

print("\nRows with any missing values:")
print(
    df_integrated[df_integrated.isna().any(axis=1)][
        [
            "date",
            "seattle_gas_price_avg",
            "seattle_gas_price_std",
            "cpi_all_items_sa",
            "cpi_core_sa",
        ]
    ].head(25)
)

print("\nFirst date with gas data:")
print(df_integrated.loc[df_integrated["seattle_gas_price_avg"].notna(), "date"].min())

print("\nMissing CPI month(s):")
print(df_integrated.loc[df_integrated["cpi_all_items_sa"].isna(), "date"])

Date range: 2002-01-01 00:00:00 to 2026-03-01 00:00:00
Duplicate dates: 0

Columns:
['date', 'period_end', 'transit_available_at', 'transit_series_id', 'transit_source_name', 'upt', 'vrm', 'vrh', 'voms', 'gas_available_at', 'gas_series_id', 'gas_source_name', 'seattle_gas_price_avg', 'seattle_gas_price_std', 'weekly_obs_count', 'inflation_available_at', 'inflation_series_id', 'inflation_source_name', 'cpi_all_items_sa', 'cpi_core_sa', 'data_available_at']

Rows with any missing values:
          date  seattle_gas_price_avg  seattle_gas_price_std  \
0   2002-01-01                    NaN                    NaN   
1   2002-02-01                    NaN                    NaN   
2   2002-03-01                    NaN                    NaN   
3   2002-04-01                    NaN                    NaN   
4   2002-05-01                    NaN                    NaN   
5   2002-06-01                    NaN                    NaN   
6   2002-07-01                    NaN                    NaN 

In [5]:
df_integrated.loc[df_integrated["seattle_gas_price_avg"].notna(), "date"].min()

Timestamp('2003-05-01 00:00:00')